# From Microbiomes to Predictions: Genomics and AI in Animal Science

This notebook walks through a complete example of how genomic data from microbial communities can be used, with machine learning, to predict an animal phenotype — and how we can explain the model's reasoning. We will work with **rumen microbiome** samples from cattle and ask whether we can predict which animals are low or high methane emitters.

**You do not need to understand or modify any code in this notebook. Run each cell and read the outputs.**

**How to run:** Press `Shift+Enter` to run the currently selected cell, or use `Runtime > Run all` in the Colab menu to run the entire notebook from top to bottom.

Sections of this notebook reveal themselves as you go: each cell's explanation and result only appear once you've run the cell before it, so try not to skip ahead.

**What you will see:** each step has a title and a ▶ play button on its left. Click it (or press `Shift+Enter`) and the result appears below, followed by what it means and a preview of the next step. The code is hidden on purpose; to peek at it, use the cell's ⋮ menu and choose *Show code*.


## Section 1: The Biology — What Are We Looking At?

⏱ **About 15 minutes · Steps 1–7.** By the end you will have a table of 30 features for each of 120 animals, and will have seen that nothing obvious jumps out by eye.

**Ruminants** (cattle, sheep, goats) digest tough plant material using a specialised stomach chamber called the rumen, which hosts a dense community of microbes. A by-product of this microbial fermentation is **methane**, a potent greenhouse gas — so understanding what drives methane emissions in livestock matters for both agriculture and climate change.

The community of microbes living in an animal's gut is called its **microbiome**. Different animals host different microbial communities, and there is growing evidence that the composition of the rumen microbiome is linked to how much methane an animal produces.

To study this with computers, we need a way to turn a microbial community into numbers. One approach is to count characteristic short DNA sequence patterns, called **k-mers**, and group similar patterns together into a **genomic embedding** — a vector of numbers that summarises the microbial makeup of a sample.

This is the bridge between biology and machine learning.


### A concrete example: what does a k-mer actually look like?

A **k-mer** is simply a short substring of length *k* taken from a longer DNA sequence, which is itself just a string of the letters A, C, G and T. To count the k-mers in a sequence, we slide a window of length *k* along it, one letter at a time, and tally how often each pattern appears. Do this across millions of DNA reads from a sample, and the resulting counts start to characterise which microbes — and how much of each — are present.

**What to expect:** a short made-up DNA sequence, then an animation showing a 4-letter window sliding along it one position at a time. Below the sequence, a table of k-mer counts builds up live as the window passes over each one — this is exactly how k-mer counting works, just slowed down so you can watch it happen. The animation plays through once; use the player controls to replay it, step through frame by frame, or jump to any point. This is the raw ingredient behind the "genomic feature" columns you'll see throughout this notebook — though, as explained below, there's one more step between this table and those columns.


### ▶ Setup

*Run the cell below first (click ▶ or press `Shift+Enter`). It loads the tools used throughout the notebook and takes a few seconds.*

In [ ]:
#@title ▶ Setup · run this cell first {display-mode: "form"}
# Workshop notebook: "From Microbiomes to Predictions: Genomics and AI in Animal Science"
# Prof Chris Creevey, Queen's University Belfast
# Run all cells in order. No modifications required.
# All data is synthetic and generated within this notebook.

import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from matplotlib import animation
from IPython.display import display, HTML, Markdown

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

LOW_COLOR = '#2196F3'
HIGH_COLOR = '#F44336'

# ---- Larger, clearer figures (easier to read on a projector) ----
plt.rcParams.update({'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
                     'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12})

# ---- Helper that shows the explanation text between steps in a larger, clearer format ----
import re as _re, html as _html
try:
    import markdown as _md
    def _to_html(t):
        return _md.markdown(t, extensions=['sane_lists'])
except Exception:
    def _to_html(t):
        t = _html.escape(t)
        t = _re.sub(r'^### (.+)$', r'<h3>\1</h3>', t, flags=_re.M)
        t = _re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', t)
        t = _re.sub(r'\*(.+?)\*', r'<i>\1</i>', t)
        t = _re.sub(r'`(.+?)`', r'<code>\1</code>', t)
        return '<p>' + t.replace('\n\n', '</p><p>') + '</p>'

_CSS = """<style>
.wk{font-size:18px;line-height:1.6;max-width:940px;margin:10px 0 18px 0}
.wk p,.wk li{font-size:18px;line-height:1.6;margin:8px 0}
.wk h3{font-size:24px;line-height:1.3;margin:10px 0 8px 0}
.wk .box{border-left:7px solid;border-radius:6px;padding:8px 18px;margin:14px 0}
.wk .res{border-color:#2E7D4F;background:rgba(46,125,79,.13)}
.wk .dis{border-color:#E07A1F;background:rgba(224,122,31,.15)}
.wk .exp{border-color:#2196F3;background:rgba(33,150,243,.13)}
.wk .tag{font-size:13px;font-weight:700;letter-spacing:.09em;text-transform:uppercase;margin:6px 0 2px 0;opacity:.9}
.wk .next{border-top:3px dashed #8a8a8a;margin-top:26px;padding-top:6px}
.wk .nexttag{font-size:15px;color:#2E7D4F}
</style>"""

def _cap(t):
    return t[:1].upper() + t[1:]

def show(text, roles, nxt=None):
    """R = what the result means, D = discussion, N = about the next step, X = what to expect."""
    blocks = [b for b in _re.split(r'\n\s*\n', text.strip()) if b.strip()]
    pick = lambda r: [b for b, x in zip(blocks, roles) if x == r]
    res, dis, nxt_b, exp = pick('R'), pick('D'), pick('N'), pick('X')
    out = ['<div class="wk">']
    if res:
        out.append('<div class="box res"><div class="tag">What this result means</div>'
                   + ''.join(_to_html(b) for b in res) + '</div>')
    if dis:
        out.append('<div class="box dis"><div class="tag">Discuss with your neighbour</div>'
                   + ''.join(_to_html(_re.sub(r'^>\s*\*\*Discussion:\*\*\s*', '', b)) for b in dis) + '</div>')
    if nxt_b or exp:
        out.append('<div class="next"><div class="tag nexttag">&#9660; Up next' + (' &middot; ' + nxt if nxt else '') + '</div>')
        out += [_to_html(b) for b in nxt_b]
        if exp:
            out.append('<div class="box exp"><div class="tag">What to expect</div>'
                       + ''.join(_to_html(_cap(_re.sub(r'^\*\*What to expect:\*\*\s*', '', b))) for b in exp) + '</div>')
        out.append('</div>')
    out.append('</div>')
    display(HTML(_CSS + ''.join(out)))

### ▶ Step 1 of 13 · See a k-mer in action

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 1 of 13 · See a k-mer in action {display-mode: "form"}
# A short illustrative DNA sequence (for teaching purposes only — not connected to the
# synthetic dataset used later in this notebook).
example_sequence = 'ATCGGCTAGCATCGGCTTAGGATCGGCTAGCTTAGGATCGGCATCGTTAGGCATCGGCTA'
k = 4

print('Example DNA sequence:')
print(example_sequence)

# Animate a length-k window sliding along the sequence one letter at a time. Each frame
# recomputes the k-mer counts from scratch up to that window position, so the table below
# genuinely tallies k-mers one at a time as the window passes over them, and scrubbing the
# player to any frame always shows the correct running total for that point.
n_windows = len(example_sequence) - k + 1
top_n_table = 10

fig, (ax_seq, ax_table) = plt.subplots(
    2, 1, figsize=(9, 6.5), dpi=100, gridspec_kw={'height_ratios': [1, 2.4]}
)

ax_seq.set_xlim(-1, len(example_sequence))
ax_seq.set_ylim(-0.6, 1)
ax_seq.axis('off')

for i, ch in enumerate(example_sequence):
    ax_seq.text(i, 0.5, ch, family='monospace', fontsize=14, ha='center', va='center')

window_patch = plt.Rectangle((-0.5, 0.15), k, 0.7, facecolor='#FFEB3B', alpha=0.5,
                              edgecolor='#F9A825', linewidth=2)
ax_seq.add_patch(window_patch)

window_label = ax_seq.text(len(example_sequence) / 2, -0.25, '', ha='center', va='center',
                            fontsize=12, family='monospace', color='#1565C0')

def update(frame):
    window_patch.set_x(frame - 0.5)
    current_kmer = example_sequence[frame:frame + k]
    window_label.set_text(f'Window {frame + 1} of {n_windows}:  k-mer = "{current_kmer}"')

    running_counts = collections.Counter(example_sequence[j:j + k] for j in range(frame + 1))
    top_running = running_counts.most_common(top_n_table)

    ax_table.clear()
    ax_table.axis('off')
    ax_table.set_title(f'Running k-mer counts (after {frame + 1} of {n_windows} windows)', fontsize=11)
    cell_text = [[kmer, str(count)] for kmer, count in top_running]
    table = ax_table.table(cellText=cell_text, colLabels=[f'{k}-mer', 'Count'],
                            loc='center', cellLoc='center', colWidths=[0.3, 0.2])
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.4)
    for row_idx, (kmer, _) in enumerate(top_running, start=1):
        color = '#FFECB3' if kmer == current_kmer else 'white'
        for col_idx in range(2):
            table[(row_idx, col_idx)].set_facecolor(color)

    return window_patch, window_label

anim = animation.FuncAnimation(fig, update, frames=n_windows, interval=250, blit=False, repeat=False)
plt.close(fig)
display(HTML(anim.to_jshtml(default_mode='once')))

show('''In a real rumen microbiome sample, this process runs on millions of DNA reads rather than one 60-letter sequence, producing thousands of distinct k-mer counts per animal — far too many columns to work with directly, and mostly redundant with one another. That huge k-mer count table is what gets compressed into the much smaller **genomic embedding**: a set of derived numbers (30, in this notebook) that summarise the same underlying information. Those 30 derived numbers — not the raw k-mer counts themselves — are the "genomic feature" columns used for the rest of this notebook.

### How do we know which microbes these k-mers belong to?

Counting k-mers is only half the story. On its own, a table of "ATCG appeared 6 times" doesn't tell us anything about biology — we need to connect it to known microbes. The standard approach: build a reference database of k-mer profiles from thousands of already-sequenced, known bacterial genomes, then compare a new sample's k-mer profile against each of them. Whichever reference profile looks most similar is the most likely match.

"Similar" here means mathematically similar — treating each genome's k-mer counts as a long vector of numbers (one entry per possible k-mer) and measuring how closely two vectors point in the same direction. This works because it has been shown that different bacterial species have **characteristically different k-mer profiles**, largely reflecting real underlying biology: **GC content** (the proportion of G/C versus A/T bases, which varies widely and consistently between species), codon usage bias, and other genome-wide sequence composition patterns. Species with similar biology tend to have similar k-mer profiles; species with very different biology don't.

**What to expect:** three short synthetic "reference genomes" built with different GC content, standing in for three different known bacterial species. Then an "unknown" sequence — deliberately built to resemble one of the three — is compared against all three reference profiles using a similarity score. A bar chart shows the similarity to each reference, and the correct match should score clearly highest.''', 'RNNNX', 'Step 2 of 13')


### ▶ Step 2 of 13 · Compare k-mer profiles to known genomes

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 2 of 13 · Compare k-mer profiles to known genomes {display-mode: "form"}
import itertools
import random

def kmer_profile(sequence, k):
    counts = collections.Counter(sequence[i:i + k] for i in range(len(sequence) - k + 1))
    total = sum(counts.values())
    all_kmers = [''.join(p) for p in itertools.product('ACGT', repeat=k)]
    return np.array([counts.get(kmer, 0) / total for kmer in all_kmers])

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Build short synthetic "reference genomes" with different GC content - a real biological
# property that varies systematically between bacterial species and shapes their k-mer
# profile. A separate, independently-seeded random generator is used here so it doesn't
# disturb the main dataset generated later in this notebook.
local_random = random.Random(7)

def make_genome(length, gc_content, rng):
    seq_chars = []
    for _ in range(length):
        if rng.random() < gc_content:
            seq_chars.append(rng.choice('GC'))
        else:
            seq_chars.append(rng.choice('AT'))
    return ''.join(seq_chars)

reference_genomes = {
    'Genome A (~35% GC)': make_genome(400, 0.35, local_random),
    'Genome B (~50% GC)': make_genome(400, 0.50, local_random),
    'Genome C (~65% GC)': make_genome(400, 0.65, local_random),
}

# An "unknown" sample sequence, deliberately built to resemble Genome C
unknown_sequence = make_genome(200, 0.65, local_random)

reference_profiles = {name: kmer_profile(seq, k) for name, seq in reference_genomes.items()}
unknown_profile = kmer_profile(unknown_sequence, k)

similarities = {name: cosine_similarity(unknown_profile, profile) for name, profile in reference_profiles.items()}
similarity_series = pd.Series(similarities).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
colors = [HIGH_COLOR if name == similarity_series.index[0] else '#90A4AE' for name in similarity_series.index]
ax.barh(similarity_series.index[::-1], similarity_series.values[::-1], color=colors[::-1])
ax.set_xlabel('Cosine Similarity of k-mer Profile to the Unknown Sequence')
ax.set_title('Matching an Unknown Sequence to Reference Genomes by k-mer Profile')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

best_match = similarity_series.index[0]
print(f'Best match: {best_match}  (similarity = {similarity_series.iloc[0]:.3f})')

show('''The unknown sequence matched its true source most closely, simply by comparing k-mer profiles — no alignment, no gene annotation, nothing biological beyond counting short substrings and measuring vector similarity. Scale this up to thousands of reference genomes and millions of reads per sample, and this is essentially how real metagenomic classification tools (e.g. Kraken, Centrifuge) assign reads to species.

### So what does a "genomic embedding" actually look like?

We've been saying k-mer counts get "compressed into a genomic embedding" — but so far that's just been a phrase. Let's make it concrete. For k=4 there are 4×4×4×4 = 256 possible k-mers (AAAA, AAAC, AAAG, ... TTTT). A full k-mer profile — like the ones just used to match genomes above — is nothing more than a vector of 256 numbers: one relative frequency per possible k-mer, always in the same order, so any two profiles can be compared position-by-position.

That raw 256-number vector is already a crude embedding, but it's large and sparse (most 4-letter combinations barely appear in a short sequence), which makes it noisy to compare across many samples. In practice, this raw profile is compressed further — down to a much smaller number of dimensions that summarise the same information more robustly. That's exactly what the 30 "genomic feature" columns you're about to meet are: not raw k-mer counts, but a compressed numerical fingerprint, one number per dimension, computed this same way.

**What to expect:** first, the actual 256-number k-mer profile vector for Genome C, so you can see the raw ingredient. Then that vector, and the other genomes' profiles, compressed down to just 3 numbers each using PCA (the same technique used in Section 2) — a small table where each row is a genome and each column is one "embedding dimension". This is precisely the kind of vector that stands behind every animal in the main dataset, just with 30 dimensions instead of 3.''', 'RNNNX', 'Step 3 of 13')


### ▶ Step 3 of 13 · See a genomic embedding as a vector of numbers

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 3 of 13 · See a genomic embedding as a vector of numbers {display-mode: "form"}
all_kmers = [''.join(p) for p in itertools.product('ACGT', repeat=k)]
genome_c_profile = reference_profiles['Genome C (~65% GC)']

print(f'Genome C k-mer profile: a vector of {len(genome_c_profile)} numbers (one per possible {k}-mer)')
print()
preview_df = pd.DataFrame({
    f'{k}-mer': all_kmers[:16],
    'Relative frequency': np.round(genome_c_profile[:16], 4),
})
display(preview_df)
print(f'... and {len(genome_c_profile) - 16} more values not shown. As a raw array, it starts like this:')
print(np.round(genome_c_profile[:16], 3))

# Compress the 256-dimensional k-mer profiles down to a small, dense "genomic embedding"
# using PCA - exactly the same compression idea used for the main 30-feature dataset later,
# just applied here, in miniature, to genomes whose profiles we can actually inspect.
profile_matrix = np.vstack(list(reference_profiles.values()) + [unknown_profile])
profile_labels = list(reference_profiles.keys()) + ['Unknown sequence']

embedding_pca = PCA(n_components=3, random_state=42)
embeddings = embedding_pca.fit_transform(profile_matrix)

embedding_df = pd.DataFrame(
    np.round(embeddings, 3),
    columns=[f'embedding_dim_{i + 1}' for i in range(3)],
    index=profile_labels,
)
print()
print('Compressed genomic embeddings (3 dimensions, derived from the 256-number k-mer profiles):')
display(embedding_df)

show('''Each row above is a genomic embedding: a short, fixed-length vector of plain numbers, with no biological labels attached to any individual dimension — you can't point to `embedding_dim_2` and say "that's Methanobrevibacter". The 30 "genomic feature" columns used for the rest of this notebook work exactly the same way in principle. The difference is scale: real genomic embeddings are typically built from many more k-mers, many more reference genomes, and often a more powerful compression step than the simple PCA used here — but the core idea is identical, and it's exactly what "the bridge between biology and machine learning" from Section 1 actually means in practice.

> **Discussion:** Before we look at the data, what do you think the microbiome of a low methane cow might look like compared to a high methane cow? What types of microbes might be different?

### How diverse is each animal's microbiome?

Every sample contains dozens of different microbial signals, but usually only a handful dominate any one animal. To get a feel for this diversity, we can turn each sample's genomic feature values into proportions — how much of that animal's profile is made up of its most common microbial groups — and stack them into a single bar per animal. The 8 largest groups (by average proportion across the dataset) are shown individually; everything else is lumped into **Other**.

**What to expect:** two stacked bar charts, one for high emitters and one for low emitters. Each vertical bar is one animal, and the coloured segments show the proportion of each microbial group present in it. You should see broadly similar mixes across animals, dominated by the same handful of groups, with the exact proportions varying from animal to animal. That variation is normal biological diversity — and part of why finding the phenotype signal isn't as simple as eyeballing composition.''', 'RDNNX', 'Step 4 of 13')


### ▶ Step 4 of 13 · Explore microbiome diversity

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 4 of 13 · Explore microbiome diversity {display-mode: "form"}
N = 120          # total samples
M = 30           # genomic features
N_TRAIN_FRAC = 80 / 120

# Named genomic features. The first 10 carry biological signal;
# the remaining 20 are generic/noise features.
feature_names = [
    'Butyrivibrio_cluster',        # cellulolytic, competes with methanogens -> LOW emitters
    'Methanobrevibacter_cluster',  # primary methanogen -> HIGH emitters
    'Ruminococcus_cluster',        # fibre degrader, competes with methanogens -> LOW emitters
    'Prevotella_cluster',          # generalist fermenter, moderate -> LOW emitters
    'Fibrobacter_cluster',         # cellulolytic, competes with methanogens -> LOW emitters
    'Methanomassiliicoccus_cluster', # secondary methanogen -> HIGH emitters
    'Lachnospiraceae_cluster',     # noise
    'Treponema_cluster',           # noise
    'Succiniclasticum_cluster',    # noise
    'Clostridium_cluster',         # noise
] + [f'genomic_feature_{i}' for i in range(11, 31)]

assert len(feature_names) == M

# Phenotype labels: ~60/40 low/high split
n_low = int(round(N * 0.6))
n_high = N - n_low
labels = np.array([0] * n_low + [1] * n_high)  # 0 = low_emitter, 1 = high_emitter
np.random.shuffle(labels)

# Continuous phenotype: methane emissions (g/day)
methane = np.where(
    labels == 0,
    np.random.normal(180, 25, size=N),
    np.random.normal(310, 35, size=N),
)

# Build genomic feature matrix.
# Biological rationale for each signal:
#   Butyrivibrio, Fibrobacter, Ruminococcus: efficient fibre-fermenting bacteria that
#       outcompete methanogens for hydrogen -> elevated in LOW emitters.
#   Methanobrevibacter, Methanomassiliicoccus: methanogenic archaea that directly
#       produce methane -> elevated in HIGH emitters.
#   Prevotella: generalist fermenter loosely associated with more efficient,
#       lower-emission fermentation pathways -> mildly elevated in LOW emitters.
#   All remaining features: no true phenotype association, included as biological
#       and technical noise (as in real microbiome data).
X = np.random.normal(loc=0.0, scale=1.0, size=(N, M))

low_boost = {'Butyrivibrio_cluster': 1.4, 'Fibrobacter_cluster': 1.2, 'Ruminococcus_cluster': 1.3,
             'Prevotella_cluster': 0.6}
high_boost = {'Methanobrevibacter_cluster': 1.6, 'Methanomassiliicoccus_cluster': 1.0}

for name, boost in low_boost.items():
    idx = feature_names.index(name)
    X[labels == 0, idx] += boost

for name, boost in high_boost.items():
    idx = feature_names.index(name)
    X[labels == 1, idx] += boost

# Turn genomic feature values into a "community composition" for this plot only:
# convert to positive weights and normalise each sample to sum to 1, giving a
# proportion per microbial group, similar to relative-abundance profiles in real
# microbiome studies. The 8 groups with the largest average proportion are shown
# individually; the rest are pooled into "Other".
composition_weights = np.exp(X)
composition_proportions = composition_weights / composition_weights.sum(axis=1, keepdims=True)

top_n = 8
mean_proportion = composition_proportions.mean(axis=0)
top_idx = np.argsort(mean_proportion)[::-1][:top_n]
other_idx = np.array([i for i in range(M) if i not in top_idx])

composition_labels = [feature_names[i] for i in top_idx] + ['Other']
composition_matrix = np.column_stack(
    [composition_proportions[:, i] for i in top_idx] + [composition_proportions[:, other_idx].sum(axis=1)]
)

group_colors = list(plt.get_cmap('tab10').colors[:top_n]) + ['#B0BEC5']

fig, axes = plt.subplots(2, 1, figsize=(10, 9), dpi=100)

for ax, phenotype_val, phenotype_name in [(axes[0], 1, 'High Emitters'), (axes[1], 0, 'Low Emitters')]:
    mask = labels == phenotype_val
    comp = composition_matrix[mask]
    x_pos = np.arange(comp.shape[0])
    bottom = np.zeros(comp.shape[0])
    for j, group_label in enumerate(composition_labels):
        ax.bar(x_pos, comp[:, j], bottom=bottom, width=1.0, color=group_colors[j],
               label=group_label if ax is axes[0] else None)
        bottom += comp[:, j]
    ax.set_title(f'Microbial Community Composition: {phenotype_name} (n={mask.sum()})')
    ax.set_xlabel('Sample')
    ax.set_ylabel('Proportion of Community')
    ax.set_xlim(-0.5, comp.shape[0] - 0.5)
    ax.set_ylim(0, 1)

axes[0].legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, title='Microbial Group')
plt.tight_layout()
plt.show()

show('''Look closely and you may notice the high-emitter chart leaning more heavily on some colours than the low-emitter chart does — but every animal is still a different mix, and both charts share the same broad palette of groups. Any consistent difference between phenotypes is a shift in proportions, not a clean separation — which is exactly what the rest of this notebook is designed to uncover.

### A first look: average abundance by phenotype

The composition charts above hinted at a subtle shift between phenotypes, but with so many overlapping colours in each bar, it's hard to say exactly which microbial groups are driving it. A more direct question: taken one at a time, is each genomic feature typically higher in low emitters, higher in high emitters, or about the same in both? Let's average each feature separately across the two groups and compare.

**What to expect:** a bar chart with two bars per genomic feature — one for the low-emitter average, one for the high-emitter average. For most of the 30 features, the two bars will be almost the same height: no real difference. But for a handful of features, you may be able to spot a visible gap between the blue and red bars. Keep an eye out for which ones those are.''', 'RNNX', 'Step 5 of 13')


### ▶ Step 5 of 13 · Compare average abundance by phenotype

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 5 of 13 · Compare average abundance by phenotype {display-mode: "form"}
feature_means_low = X[labels == 0].mean(axis=0)
feature_means_high = X[labels == 1].mean(axis=0)

y_pos = np.arange(len(feature_names))
bar_height = 0.4

fig, ax = plt.subplots(figsize=(8, 10), dpi=100)
ax.barh(y_pos - bar_height / 2, feature_means_low, height=bar_height, color=LOW_COLOR, label='Low Emitter (average)')
ax.barh(y_pos + bar_height / 2, feature_means_high, height=bar_height, color=HIGH_COLOR, label='High Emitter (average)')
ax.set_yticks(y_pos)
ax.set_yticklabels(feature_names, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Average Genomic Feature Value (relative to cohort average)')
ax.set_title('Average Relative Abundance per Genomic Feature, by Phenotype')
ax.legend()
plt.tight_layout()
plt.show()

show('''> **Discussion:** Do any of the genomic features show a visible gap between the low- and high-emitter averages? Which ones — and does that match what you might have predicted before seeing the data?

### Now zoom in: the raw, individual values

The bar chart above shows *averages*, computed across dozens of samples — that averaging is what let any pattern show through. But no single animal is "average". Below is a histogram of every individual genomic feature value in the dataset, before any averaging, split by phenotype.

**What to expect:** the two distributions will overlap almost completely — from this view alone you cannot tell which numbers belong to which group. That's the real challenge: the biological signal is genuine, but at the level of a single sample it's subtle and spread thinly across many correlated features, not sitting in one obvious place.''', 'DNNX', 'Step 6 of 13')


### ▶ Step 6 of 13 · Zoom in on individual values

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 6 of 13 · Zoom in on individual values {display-mode: "form"}
fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
ax.hist(X[labels == 0].flatten(), bins=40, alpha=0.6, label='Low Emitter', color=LOW_COLOR, density=True)
ax.hist(X[labels == 1].flatten(), bins=40, alpha=0.6, label='High Emitter', color=HIGH_COLOR, density=True)
ax.set_xlabel('Genomic Feature Value (relative to cohort average)')
ax.set_ylabel('Density')
ax.set_title('Distribution of All 3,600 Genomic Feature Values, by Phenotype')
ax.legend()
plt.tight_layout()
plt.show()

show('''As expected, the two distributions sit almost exactly on top of each other. Pooling all individual values together like this hides any structure — most of these numbers really are just noise, and the handful of genuinely informative features are diluted among them. To find the signal, we need to look at the data more carefully, starting with the individual samples themselves.

### What the data looks like

In reality, a dataset like this starts as raw DNA sequencing reads, which are first processed into **k-mer counts** and then summarised into the genomic embedding described in Section 1 — that processing happens in the lab and on a computing cluster, long before anyone sees a spreadsheet. What you see below is that finished genomic embedding: the starting point for *this* notebook, not raw sequencer output. Nothing has been transformed, reduced, or modelled any further than that. Each row is one animal, with:

- **sample_id** — a unique identifier for the animal
- **phenotype** — whether the animal is a `low_emitter` or `high_emitter`, based on its measured methane output
- **methane_g_per_day** — the animal's measured methane emissions, in grams per day
- **genomic feature columns** (e.g. `Butyrivibrio_cluster`, `Methanobrevibacter_cluster`) — one column per microbial genomic signature. Each value is a relative abundance score, scaled so that 0 is the **average across all 120 samples in this dataset** for that feature (much like a z-score). Positive means more abundant than the cohort average, negative means less abundant — the scale is relative to this group of animals, not an absolute biological unit

To keep the table readable we show only the first 10 of 120 samples, and the first 5 of 30 genomic features — but this is the full embedding, not a further summary or analysis of it.

**What to expect:** a plain data table. Scanning it by eye, the low and high emitters will **not** look obviously different — same story as the histogram above, just at the level of individual samples instead of pooled values.''', 'RNNNNX', 'Step 7 of 13')


### ▶ Step 7 of 13 · View the dataset

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 7 of 13 · View the dataset {display-mode: "form"}
sample_ids = [f'Sample_{i+1:03d}' for i in range(N)]

df = pd.DataFrame(X, columns=feature_names)
df.insert(0, 'sample_id', sample_ids)
df.insert(1, 'phenotype', np.where(labels == 0, 'low_emitter', 'high_emitter'))
df.insert(2, 'methane_g_per_day', np.round(methane, 1))

display_cols = ['sample_id', 'phenotype', 'methane_g_per_day'] + feature_names[:5]
display(df[display_cols].head(10))

show('''As expected, nothing jumps out from scanning the raw numbers either — with 30 features per sample, the differences between groups are too spread out to see by eye. The next section starts making sense of this.

We've now looked at the raw data from several angles — composition, feature averages, a pooled histogram, a table of individual samples — and each time, low and high emitters looked essentially the same at a glance. That's because each sample has 30 numbers describing it — too many dimensions for a human eye to compare directly. **PCA (Principal Component Analysis)** is a technique built exactly for this problem: it compresses many dimensions into two, while preserving as much of the variation in the data as possible.

If low and high emitters really do differ in their microbiome, we would expect to see them separate when plotted in this 2D space. This is not a biological method — it is a mathematical one. We are simply asking: does the biology show up in the maths?

**What to expect:** a scatter plot with two colours — blue for low emitters, red for high emitters. They probably won't separate perfectly (this is noisy biological data), but you should be able to see a general trend of blue points sitting toward one side and red points toward the other.''', 'RNNX', 'Step 8 of 13')


## Section 2: Visualising the Data — Can We See the Difference?

⏱ **About 7 minutes · Steps 8–9.** *Checkpoint: you should be here by about minute 15.*

### ▶ Step 8 of 13 · Visualise with PCA

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 8 of 13 · Visualise with PCA {display-mode: "form"}
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
var_explained = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
for label_val, name, color in [(0, 'Low Emitter', LOW_COLOR), (1, 'High Emitter', HIGH_COLOR)]:
    mask = labels == label_val
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=name, s=50, alpha=0.75, edgecolor='white')

ax.set_xlabel(f'Principal Component 1 ({var_explained[0]:.1f}% variance explained)')
ax.set_ylabel(f'Principal Component 2 ({var_explained[1]:.1f}% variance explained)')
ax.set_title('PCA of Rumen Microbiome Genomic Embeddings')
ax.legend()
plt.tight_layout()
plt.show()

show('''The scatter plot above showed some separation along PC1 — but PC1 itself is just a mathematical combination of all 30 original features, each weighted differently. To connect this back to biology, we can ask: which features contribute most to PC1? Those weights are called **loadings**, and the largest ones tell us what's actually driving the separation we just saw.

**What to expect:** a horizontal bar chart of the 10 features with the strongest pull on PC1. You should see methanogen-associated features (e.g. `Methanobrevibacter_cluster`) pointing one direction, and fibre-fermenting features (e.g. `Butyrivibrio_cluster`, `Fibrobacter_cluster`) pointing the other — matching the biology from Section 1.''', 'NX', 'Step 9 of 13')


### ▶ Step 9 of 13 · See what drives the separation

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 9 of 13 · See what drives the separation {display-mode: "form"}
loadings = pd.Series(pca.components_[0], index=feature_names)
top_loadings = loadings.reindex(loadings.abs().sort_values(ascending=False).index).head(10)

fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
colors = [LOW_COLOR if v < 0 else HIGH_COLOR for v in top_loadings.values]
ax.barh(top_loadings.index[::-1], top_loadings.values[::-1], color=colors[::-1])
ax.set_xlabel('Loading on Principal Component 1')
ax.set_title('Top 10 Genomic Features Driving PC1')
plt.tight_layout()
plt.show()

show('''> **Discussion:** Do the low and high emitters separate clearly? What would it mean if they overlapped completely?

> **Discussion:** Which microbial features appear to drive the separation? Do these make biological sense based on what you read in the introduction?

PCA hinted that combining features mathematically can reveal some separation between the groups — but it only used 2 of the 30 dimensions, and it wasn't even trying to predict the phenotype, just summarise the variation. The natural next question: can a model that uses all 30 features, and is actually trained to predict the phenotype, do better?

A **classifier** is a mathematical model that learns to assign a label (low or high emitter) based on input data. We train it on some samples and test it on samples it has never seen — this tells us whether it has genuinely learned something useful, or simply memorised the training data.

We will use a **Random Forest**: an ensemble of many decision trees, each making simple yes/no splits on the features. You do not need to understand the maths — focus on what it learns and how well it performs.

**What to expect:** a printed accuracy somewhere in the 80–90% range — not 100%, because this is realistic, noisy biological data — followed by a confusion matrix that is mostly on the diagonal with a handful of misclassifications, and a small table of precision and recall for each class.''', 'DDNNNX', 'Step 10 of 13')


## Section 3: Training a Classifier — Can a Model Learn This?

⏱ **About 10 minutes · Steps 10–11 and the bonus.** *Checkpoint: you should be here by about minute 22.*

### ▶ Step 10 of 13 · Train a classifier

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 10 of 13 · Train a classifier {display-mode: "form"}
X_train, X_test, y_train, y_test, methane_train, methane_test, idx_train, idx_test = train_test_split(
    X, labels, methane, np.arange(N), test_size=0.2, random_state=4, stratify=labels
)

clf = RandomForestClassifier(n_estimators=100, random_state=4)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred) * 100
print(f'Accuracy: {accuracy:.1f}%')

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Emitter', 'High Emitter'],
            yticklabels=['Low Emitter', 'High Emitter'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix: Random Forest Classifier')
plt.tight_layout()
plt.show()

precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1])
metrics_df = pd.DataFrame({
    'Class': ['Low Emitter', 'High Emitter'],
    'Precision': np.round(precision, 2),
    'Recall': np.round(recall, 2),
})
display(metrics_df)

show('''The accuracy and confusion matrix above tell us *how often* the model was right, but not *why* it got the rest wrong. A natural next question: are those mistakes scattered randomly, or do they happen for animals whose true methane output sits close to the boundary between "low" and "high"? Let's compare the actual methane values of the samples the model classified correctly against the ones it got wrong.

**What to expect:** two overlapping histograms of methane values — one for correctly classified samples, one for misclassified samples. If the model is doing something sensible, the misclassified samples should cluster around the middle of the methane range, where low and high emitters are genuinely hard to tell apart.''', 'NX', 'Step 11 of 13')


### ▶ Step 11 of 13 · Check where the model struggles

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 11 of 13 · Check where the model struggles {display-mode: "form"}
correct = y_pred == y_test
plot_df = pd.DataFrame({
    'methane': methane_test,
    'correct': np.where(correct, 'Correctly Classified', 'Misclassified'),
})

n_correct = int(correct.sum())
n_misclassified = int((~correct).sum())
print(f'{n_correct} correctly classified, {n_misclassified} misclassified (out of {len(correct)} test samples)')

# Shared bin edges so both groups are compared on the same scale, and both groups are
# always plotted (even if empty) so the legend never silently drops a category - if a run
# has zero misclassifications, that will say so explicitly rather than just disappearing.
bin_edges = np.linspace(methane_test.min(), methane_test.max(), 13)

fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
for group, color in [('Correctly Classified', '#4CAF50'), ('Misclassified', '#9E9E9E')]:
    subset = plot_df[plot_df['correct'] == group]['methane']
    ax.hist(subset, bins=bin_edges, alpha=0.6, label=f'{group} (n={len(subset)})', color=color)

ax.set_xlabel('Methane Emissions (g/day)')
ax.set_ylabel('Number of Test Samples')
ax.set_title('Methane Emissions: Correctly vs Incorrectly Classified Samples')
ax.legend()
plt.tight_layout()
plt.show()

show('''Every result above used a *fixed* random seed, so it's perfectly reproducible — you and your neighbour will always see exactly the same numbers. But underneath that fixed seed, splitting into training and test sets is still fundamentally a random draw of which animals end up where. A different draw could easily give a different accuracy, just by chance — especially with only 24 test samples.

### 🎲 Bonus: how much does the random split matter?

**What to expect:** an accuracy and a misclassified count, with no fixed seed this time. Run the cell below more than once and watch the numbers move around a little from run to run — a concrete reminder that a single accuracy figure, from a single split, is only ever an estimate.''', 'NNX', 'Bonus')


### 🎲 Bonus · Run this cell again and again

*Run the cell below, then run it again.*

In [ ]:
#@title ▶ Bonus · Run this cell again and again {display-mode: "form"}
# No fixed random_state here on purpose: this cell gives a different train/test split,
# and a freshly trained model, every time you run it.
bonus_X_train, bonus_X_test, bonus_y_train, bonus_y_test = train_test_split(
    X, labels, test_size=0.2, stratify=labels
)

bonus_clf = RandomForestClassifier(n_estimators=100)
bonus_clf.fit(bonus_X_train, bonus_y_train)
bonus_pred = bonus_clf.predict(bonus_X_test)

bonus_accuracy = accuracy_score(bonus_y_test, bonus_pred) * 100
bonus_correct = bonus_pred == bonus_y_test
bonus_n_correct = int(bonus_correct.sum())
bonus_n_misclassified = int((~bonus_correct).sum())

print(f'This run: {bonus_accuracy:.1f}% accuracy — {bonus_n_correct} correct, {bonus_n_misclassified} misclassified '
      f'(out of {len(bonus_y_test)} test samples)')
print('Run this cell again (Shift+Enter) to see a different random split.')

show('''> **Discussion:** The model is not perfect. What are the consequences of a false positive (predicting low emitter when the animal is actually high)? What about a false negative?

> **Discussion:** The model was trained on 96 samples. Do you think this is enough? What might change with 10,000 samples?

A model that makes accurate predictions is useful, but in science and in real-world applications we also need to know *why* it made a prediction. Random Forests provide a built-in measure called **feature importance** — which input features did the model rely on most heavily?

This is a simple form of **explainable AI (XAI)**: making the model's reasoning transparent. If the most important features make biological sense, that gives us more confidence in the model — and generates hypotheses for further research.

**What to expect:** a horizontal bar chart of the top 15 features by importance. The biologically relevant features (highlighted in orange) should tend to rank near the top — a sign the model has picked up on real biological signal rather than noise.''', 'DDNNX', 'Step 12 of 13')


## Section 4: Explainability — Why Did the Model Decide That?

⏱ **About 8 minutes · Steps 12–13.** *Checkpoint: you should be here by about minute 32.*

### ▶ Step 12 of 13 · Rank feature importance

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 12 of 13 · Rank feature importance {display-mode: "form"}
biological_features = set(low_boost.keys()) | set(high_boost.keys())

importances = pd.Series(clf.feature_importances_, index=feature_names)
top_importances = importances.sort_values(ascending=False).head(15)

colors = ['#FF9800' if f in biological_features else '#B0BEC5' for f in top_importances.index]

fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
ax.barh(top_importances.index[::-1], top_importances.values[::-1], color=colors[::-1])
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 Features by Random Forest Importance')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#FF9800', label='Biologically relevant feature'),
                    Patch(facecolor='#B0BEC5', label='Other feature')]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

show('''The chart above shows which features matter most *overall*, averaged across every prediction the model makes — but it doesn't tell us what happened for any one animal. To make this concrete, let's zoom into two individual test samples — one correctly predicted as a low emitter, one as a high emitter — and see exactly which features tipped the model's decision for each of them.

**What to expect:** two side-by-side bar charts — one for a correctly classified low emitter, one for a correctly classified high emitter — each showing the 5 features that contributed most to that individual prediction.''', 'NX', 'Step 13 of 13')


### ▶ Step 13 of 13 · Explain individual predictions

*Run the cell below: click ▶ or press `Shift+Enter`.*

In [ ]:
#@title ▶ Step 13 of 13 · Explain individual predictions {display-mode: "form"}
low_idx = [i for i, (p, c) in enumerate(zip(y_pred, correct)) if p == 0 and c][0]
high_idx = [i for i, (p, c) in enumerate(zip(y_pred, correct)) if p == 1 and c][0]

def top_features_for_sample(sample_row, importances, n=5):
    contribution = sample_row * importances.values
    contrib_series = pd.Series(contribution, index=importances.index)
    return contrib_series.reindex(contrib_series.abs().sort_values(ascending=False).index).head(n)

low_sample_contrib = top_features_for_sample(X_test[low_idx], importances)
high_sample_contrib = top_features_for_sample(X_test[high_idx], importances)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=100)

axes[0].barh(low_sample_contrib.index[::-1], low_sample_contrib.values[::-1], color=LOW_COLOR)
axes[0].set_title('Why the model said LOW EMITTER')
axes[0].set_xlabel('Contribution (feature value x importance)')

axes[1].barh(high_sample_contrib.index[::-1], high_sample_contrib.values[::-1], color=HIGH_COLOR)
axes[1].set_title('Why the model said HIGH EMITTER')
axes[1].set_xlabel('Contribution (feature value x importance)')

plt.tight_layout()
plt.show()

show('''> **Discussion:** The model identified Methanobrevibacter as a key predictor of high emissions. Does this mean Methanobrevibacter *causes* high emissions? How would you test this?

> **Discussion:** If a model is accurate but its top features make no biological sense, what would you conclude?''', 'DD', None)


## Section 5: What This Means — and What It Doesn't

⏱ **About 3 minutes.** *Checkpoint: you should be here by about minute 40.*

We have demonstrated that a genomic representation of a microbial community carries **predictive signal** for a host phenotype, and that a simple machine learning model can extract and explain that signal in a way that aligns with known biology.

What this does **not** prove is causality, generalisability across different breeds, diets, or environments, or readiness for use as a real-world management tool. Real microbiome research faces further challenges: small sample sizes, **batch effects** across different studies, **compositional data** constraints (relative abundances that must sum to a fixed total), and the difficulty of integrating multiple independent datasets.

This is exactly the kind of question that motivated research programmes such as James Barnard's PhD — using explainable AI to understand which microbial drivers are genuinely robust across datasets, rather than artefacts of a single study.


## Section 6: Summary and Key Concepts

- **Genomic embeddings**: k-mer frequency vectors used as numerical representations of microbial communities
- **Dimensionality reduction (PCA)**: compressing many features into a small number of axes that preserve variation, for visualisation
- **Classification (Random Forest)**: an ensemble model that learns to assign labels from data, evaluated on held-out test samples
- **Feature importance**: a simple form of explainable AI (XAI) that shows which inputs a model relied on
- **Prediction vs causal explanation**: a model can predict accurately without proving *why* something happens biologically

**Further Reading**
- Roehe, R. et al. (2016) "Bovine Host Genetic Variation Influences Rumen Microbial Methane Production with Best Selection Criterion for Low Methane Emitting and Efficiently Feed Converting Hosts Based on Metagenomic Gene Abundance." *PLOS Genetics*.
- Ritchie, H. (2020) "Emissions from food alone could use up all of our budget for 1.5°C or 2°C — but this isn't the most likely outcome." Our World in Data. https://ourworldindata.org/food-emissions-carbon-budget
- Lundberg, S. M. & Lee, S.-I. (2017) "A Unified Approach to Interpreting Model Predictions." *NeurIPS*. (Background on explainable AI / feature attribution.)


## Section 7: Explore the Data — Try It Yourself

⏱ **15–25 minutes.** *Finished early? Come straight here. Choose three exercises and compare results with a neighbour.*

The sections above walked you through one path through this analysis — but almost every step involved a choice: how much data to hold back for testing, what counts as "confident enough" to call something a high emitter, how many trees to grow, which principal components to look at. Change those choices, and the results can change too.

Below are seven short, self-contained exercises, one revisiting each part of the notebook. Each has a clearly marked variable near the top of the cell — **change the value, then re-run just that cell** (click into it and press `Shift+Enter`). You don't need to re-run anything above; everything these exercises need has already been computed by the sections above, so make sure you've run those first (or use `Runtime > Run all` before starting).

There's no single right answer here — the point is to build a feel for how sensitive, or how robust, each part of this pipeline is to the choices behind it. Try each one with a neighbour and compare what you find.


### Exercise 1: Change the k-mer length

Back in Step 1, k-mers were 4 letters long. What happens with shorter or longer windows?

**Try it:** change `TRY_K` below to 2, 3, 6, or 8, and re-run the cell.

**Watch for:** how the number of *distinct* k-mers found changes. Very short k-mers repeat constantly and carry little information; very long ones become almost unique to this exact sequence and stop generalising. Real k-mer counting in genomics usually lands around k=15–31 for exactly this trade-off.


In [ ]:
# TRY IT YOURSELF: change TRY_K (try 2, 3, 6, 8), then re-run this cell.
TRY_K = 4

if not (1 <= TRY_K <= len(example_sequence)):
    print(f'TRY_K must be between 1 and {len(example_sequence)}.')
else:
    n_try_windows = len(example_sequence) - TRY_K + 1
    try_kmer_counts = collections.Counter(example_sequence[i:i + TRY_K] for i in range(n_try_windows))

    print(f'k = {TRY_K}')
    print(f'Number of {TRY_K}-mer windows: {n_try_windows}')
    print(f'Number of DISTINCT {TRY_K}-mers found: {len(try_kmer_counts)}')
    print()
    try_kmer_df = pd.DataFrame(try_kmer_counts.most_common(10), columns=[f'{TRY_K}-mer', 'Count'])
    display(try_kmer_df)


### Exercise 2: Change the "unknown" sequence's GC content

In Step 2, the unknown sequence was built with 65% GC content and matched Genome C. What happens for GC values in between the three references?

**Try it:** change `TRY_GC` below — try 0.35, 0.42, 0.50, 0.58, 0.65, or 0.9 — and re-run the cell.

**Watch for:** how confidently the sequence matches its "true" source. Values close to a reference's own GC content should match strongly; values roughly halfway between two references should produce a much closer contest between them — a concrete feel for what an "ambiguous" classification looks like.


In [ ]:
# TRY IT YOURSELF: change TRY_GC (any value from 0 to 1), then re-run this cell.
TRY_GC = 0.65

exercise_rng = random.Random(99)
try_unknown_sequence = make_genome(200, TRY_GC, exercise_rng)
try_unknown_profile = kmer_profile(try_unknown_sequence, k)

try_similarities = {name: cosine_similarity(try_unknown_profile, profile) for name, profile in reference_profiles.items()}
try_similarity_series = pd.Series(try_similarities).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4), dpi=100)
colors = [HIGH_COLOR if name == try_similarity_series.index[0] else '#90A4AE' for name in try_similarity_series.index]
ax.barh(try_similarity_series.index[::-1], try_similarity_series.values[::-1], color=colors[::-1])
ax.set_xlabel('Cosine Similarity')
ax.set_title(f'Matching a Sequence with {TRY_GC:.0%} GC Content to Reference Genomes')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

print(f'Best match: {try_similarity_series.index[0]}  (similarity = {try_similarity_series.iloc[0]:.3f})')


### Exercise 3: Inspect a single feature instead of everything pooled

Step 6 pooled all 30 features into one histogram, and the phenotypes looked identical. What if you look at just one feature at a time?

**Try it:** change `FEATURE_TO_INSPECT` below to a real biological signal (e.g. `'Methanobrevibacter_cluster'`, `'Butyrivibrio_cluster'`, `'Fibrobacter_cluster'`) or a noise feature (e.g. `'genomic_feature_14'`), and re-run the cell. The full list of names is in the table from Step 7.

**Watch for:** signal features should show visibly separated histograms; noise features shouldn't. This is exactly why pooling everything together in Step 6 hid the story that Step 5's bar chart already hinted at.


In [ ]:
# TRY IT YOURSELF: change FEATURE_TO_INSPECT to any name from feature_names, then re-run.
FEATURE_TO_INSPECT = 'Methanobrevibacter_cluster'

if FEATURE_TO_INSPECT not in feature_names:
    print(f'"{FEATURE_TO_INSPECT}" is not a valid feature name. Check the spelling against the table in Step 7.')
else:
    feature_idx = feature_names.index(FEATURE_TO_INSPECT)
    fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
    ax.hist(X[labels == 0, feature_idx], bins=20, alpha=0.6, label='Low Emitter', color=LOW_COLOR, density=True)
    ax.hist(X[labels == 1, feature_idx], bins=20, alpha=0.6, label='High Emitter', color=HIGH_COLOR, density=True)
    ax.set_xlabel(f'{FEATURE_TO_INSPECT} (relative to cohort average)')
    ax.set_ylabel('Density')
    ax.set_title(f'Distribution of {FEATURE_TO_INSPECT}, by Phenotype')
    ax.legend()
    plt.tight_layout()
    plt.show()


### Exercise 4: Look at different principal components

Steps 7–8 plotted PC1 vs PC2. Does separation show up on other pairs of components too?

**Try it:** change `PC_X` and `PC_Y` below to any pair from 1 to 5, then re-run the cell.

**Watch for:** separation is concentrated on PC1 — most other pairs should look far more mixed. PCA finds the directions of greatest *variance* in the data, not automatically the direction that's most biologically meaningful; here they happen to line up on PC1, but that's not guaranteed in general.


In [ ]:
# TRY IT YOURSELF: change PC_X and PC_Y (any pair from 1 to 5), then re-run this cell.
PC_X = 1
PC_Y = 3

TRY_PCA = PCA(n_components=5, random_state=42)
TRY_X_pca = TRY_PCA.fit_transform(X)
TRY_var_explained = TRY_PCA.explained_variance_ratio_ * 100

if not (1 <= PC_X <= 5 and 1 <= PC_Y <= 5):
    print('PC_X and PC_Y must each be between 1 and 5.')
else:
    x_idx, y_idx = PC_X - 1, PC_Y - 1
    fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
    for label_val, name, color in [(0, 'Low Emitter', LOW_COLOR), (1, 'High Emitter', HIGH_COLOR)]:
        mask = labels == label_val
        ax.scatter(TRY_X_pca[mask, x_idx], TRY_X_pca[mask, y_idx], c=color, label=name, s=50, alpha=0.75, edgecolor='white')
    ax.set_xlabel(f'Principal Component {PC_X} ({TRY_var_explained[x_idx]:.1f}% variance explained)')
    ax.set_ylabel(f'Principal Component {PC_Y} ({TRY_var_explained[y_idx]:.1f}% variance explained)')
    ax.set_title(f'PCA: PC{PC_X} vs PC{PC_Y}')
    ax.legend()
    plt.tight_layout()
    plt.show()


### Exercise 5: Retrain the classifier with different settings

Step 10 trained a Random Forest with 100 trees on 80% of the data. How much do those choices matter?

**Try it:** change `TEST_SIZE`, `N_ESTIMATORS`, and/or `MAX_DEPTH` below, then re-run the cell. Try `TEST_SIZE = 0.5` (much less training data), `N_ESTIMATORS = 1` (a single decision tree), or `MAX_DEPTH = 2` (force very shallow, "underfit" trees).

**Watch for:** accuracy dropping with less training data or a single tree; shallow trees failing to capture the pattern well. This is a hands-on look at overfitting, underfitting, and why ensembles of trees tend to beat a single one.


In [ ]:
# TRY IT YOURSELF: change TEST_SIZE, N_ESTIMATORS, and/or MAX_DEPTH, then re-run this cell.
TEST_SIZE = 0.2
N_ESTIMATORS = 100
MAX_DEPTH = None   # None = no limit; try 2, 3, or 5

Xtr, Xte, ytr, yte = train_test_split(X, labels, test_size=TEST_SIZE, random_state=42, stratify=labels)

try_clf = RandomForestClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=42)
try_clf.fit(Xtr, ytr)
try_pred = try_clf.predict(Xte)

try_accuracy = accuracy_score(yte, try_pred) * 100
print(f'Training samples: {len(Xtr)}   Test samples: {len(Xte)}')
print(f'Trees: {N_ESTIMATORS}   Max depth: {MAX_DEPTH}')
print(f'Accuracy: {try_accuracy:.1f}%')

try_cm = confusion_matrix(yte, try_pred)
fig, ax = plt.subplots(figsize=(6, 5), dpi=100)
sns.heatmap(try_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Emitter', 'High Emitter'], yticklabels=['Low Emitter', 'High Emitter'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (Your Settings)')
plt.tight_layout()
plt.show()


### Exercise 6: Change the classification threshold

Step 10's model calls a sample "high emitter" whenever it's more than 50% confident. What if that cutoff moved?

**Try it:** change `CLASSIFICATION_THRESHOLD` below — try 0.3 or 0.7 — and re-run the cell.

**Watch for:** lowering the threshold catches more true high emitters but also mislabels more low emitters as high (more false positives, fewer false negatives); raising it does the opposite. Watch precision, recall, and F1 move in opposite directions — this is the false-positive/false-negative trade-off from Step 11's discussion question, made adjustable.


In [ ]:
# TRY IT YOURSELF: change CLASSIFICATION_THRESHOLD (0 to 1; default 0.5), then re-run.
CLASSIFICATION_THRESHOLD = 0.5

test_probabilities = clf.predict_proba(X_test)[:, 1]  # probability of "high emitter"
threshold_pred = (test_probabilities > CLASSIFICATION_THRESHOLD).astype(int)

threshold_accuracy = accuracy_score(y_test, threshold_pred) * 100
threshold_precision, threshold_recall, threshold_f1, _ = precision_recall_fscore_support(
    y_test, threshold_pred, labels=[0, 1], zero_division=0
)

print(f'Threshold: {CLASSIFICATION_THRESHOLD}')
print(f'Accuracy: {threshold_accuracy:.1f}%')
print()
display(pd.DataFrame({
    'Class': ['Low Emitter', 'High Emitter'],
    'Precision': np.round(threshold_precision, 2),
    'Recall': np.round(threshold_recall, 2),
    'F1': np.round(threshold_f1, 2),
}))

threshold_cm = confusion_matrix(y_test, threshold_pred)
fig, ax = plt.subplots(figsize=(6, 5), dpi=100)
sns.heatmap(threshold_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Emitter', 'High Emitter'], yticklabels=['Low Emitter', 'High Emitter'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix at Threshold = {CLASSIFICATION_THRESHOLD}')
plt.tight_layout()
plt.show()


### Exercise 7: Explain a different sample — including a wrong one

Step 13 explained one correctly classified low emitter and one correctly classified high emitter. What does the explanation look like for a sample the model got wrong?

**Try it:** change `SAMPLE_INDEX` below to any number from 0 to 23 (there are 24 test samples), then re-run the cell. To find a misclassified one, look back at the `correct` array from Step 11, or just try a few indices.

**Watch for:** whether a misclassified sample's top features still make biological sense, or whether the model was relying on weaker, more ambiguous evidence — a concrete version of the "what if a model is accurate but its top features make no biological sense" discussion from Step 13.


In [ ]:
# TRY IT YOURSELF: change SAMPLE_INDEX (0 to 23), then re-run this cell.
SAMPLE_INDEX = 0

if not (0 <= SAMPLE_INDEX < len(X_test)):
    print(f'SAMPLE_INDEX must be between 0 and {len(X_test) - 1}.')
else:
    actual_label = 'high_emitter' if y_test[SAMPLE_INDEX] == 1 else 'low_emitter'
    predicted_label = 'high_emitter' if y_pred[SAMPLE_INDEX] == 1 else 'low_emitter'
    was_correct = 'Correct' if correct[SAMPLE_INDEX] else 'INCORRECT'

    sample_contrib = top_features_for_sample(X_test[SAMPLE_INDEX], importances)

    fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
    bar_colors = [HIGH_COLOR if v > 0 else LOW_COLOR for v in sample_contrib.values]
    ax.barh(sample_contrib.index[::-1], sample_contrib.values[::-1], color=bar_colors[::-1])
    ax.set_xlabel('Contribution (feature value x importance)')
    ax.set_title(f'Sample {SAMPLE_INDEX}: actual = {actual_label}, predicted = {predicted_label} ({was_correct})')
    plt.tight_layout()
    plt.show()
